## 학생 정보 구조화 시스템

### 출력 파서: `PydanticOutputParser` + `BaseModel`

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [9]:
from langchain_openai import ChatOpenAI

PROVIDER = "groq"

LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini", 
    },
}
MAX_RETRIES = 5  


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed, 
        max_retries=MAX_RETRIES,
    )

llm = get_llm(temperature=0.7) 

In [10]:
from pydantic import BaseModel, Field, field_validator

class StudentInfo(BaseModel):
    """학생 자기소개에서 추출한 정보"""

    name: str = Field(description="학생 이름")

    age: int | None = Field(default=None, description="나이 (숫자만). 언급이 없으면 null")

    major: str | None = Field(default=None, description="전공 학과명. 언급이 없으면 null")

    hobbies: list[str] = Field(default_factory=list, description="취미 목록. 각 항목은 짧은 명사구")

    goal: str | None = Field(default=None, description="앞으로의 목표. '~하는 것' 형태로 간결하게")

    @field_validator("age")
    @classmethod
    def check_age(cls, v):
        if v is not None and not (0 < v < 120):
            raise ValueError(f"비정상적인 나이: {v}")
        return v

In [11]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=StudentInfo)

print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "학생 자기소개에서 추출한 정보", "properties": {"name": {"description": "학생 이름", "title": "Name", "type": "string"}, "age": {"anyOf": [{"type": "integer"}, {"type": "null"}], "default": null, "description": "나이 (숫자만). 언급이 없으면 null", "title": "Age"}, "major": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "description": "전공 학과명. 언급이 없으면 null", "title": "Major"}, "hobbies": {"description": "취미 목록. 각 항목은 짧은 명사구", "items": {"type": "string"}, "title": "Hobbies", "type": "array"}, "goal": {"anyOf": [{"type": "string"}, {"type

In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 자기소개 글에서 학생 정보를 정확하게 추출하는 전문가입니다.\n"
     "규칙:\n"
     "- 글에 명시된 정보만 추출하고, 없는 정보는 추측하지 말고 null로 두세요.\n"
     "- 나이는 숫자만 쓰세요.\n"
     "- 반드시 아래 JSON 형식으로만 답하세요.\n\n"
     "{format_instructions}"),
    ("user", "자기소개: {intro}"),
]).partial(format_instructions=parser.get_format_instructions())

llm = get_llm(temperature=0)

chain = prompt | llm | parser

In [13]:
import json

intro = "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. 취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."

result = chain.invoke({"intro": intro})

print(type(result).__name__)          
print(result.name, result.age)        

print(json.dumps(result.model_dump(), ensure_ascii=False, indent=4))

StudentInfo
김민수 22
{
    "name": "김민수",
    "age": 22,
    "major": "컴퓨터공학",
    "hobbies": [
        "게임하기",
        "영화보기",
        "코딩"
    ],
    "goal": "훌륭한 개발자가 되는 것"
}


In [14]:
from langchain_core.exceptions import OutputParserException

intros = [
    intro,
    "반갑습니다, 이서연입니다. 경영학과에 다니고 있고 요즘은 테니스랑 독서에 빠져 있어요. 졸업 후에는 마케터로 일하고 싶습니다.",

    "저는 스물다섯 살 박준호라고 합니다. 기계공학 전공이고 주말마다 등산을 다녀요.",

    "제 이름은 최하늘이에요. 열아홉 살이고 그림 그리기, 피아노 치기, 고양이 돌보기를 좋아해요. 언젠가 제 이름으로 전시회를 여는 게 꿈이에요.",
]

results = chain.batch([{"intro": t} for t in intros], return_exceptions=True)

for i, r in enumerate(results, 1):
    print(f"[{i}] ", end="")
    if isinstance(r, OutputParserException):
        print(f" 파싱 실패: {str(r).splitlines()[0]}")
    elif isinstance(r, Exception):
        print(f" 오류: {r}")
    else:
        print(json.dumps(r.model_dump(), ensure_ascii=False))

[1] {"name": "김민수", "age": 22, "major": "컴퓨터공학", "hobbies": ["게임하기", "영화보기", "코딩"], "goal": "훌륭한 개발자가 되는 것"}
[2] {"name": "이서연", "age": null, "major": "경영학과", "hobbies": ["테니스", "독서"], "goal": "마케터가 되는 것"}
[3] {"name": "박준호", "age": 25, "major": "기계공학", "hobbies": ["등산"], "goal": null}
[4] {"name": "최하늘", "age": 19, "major": null, "hobbies": ["그림 그리기", "피아노 치기", "고양이 돌보기를"], "goal": "전시회를 여는 것"}
